# 🦎 Text coding with Laya

Text coding assigns a category or a value to each passage of a text. Delach et al. coded the
listing documents for 459 species by hand, recording for each one whether climate change was named
as a threat and whether any action to address it was planned. A **coding scheme** is the set of
questions we ask of each passage and the answers we allow.

[Laya](https://huggingface.co/convaiinnovations/laya) applies a coding scheme that we write. Its
answers tell us as much about our scheme as about the text, so the scheme is the thing we design
and test.

## Question types

Laya answers three types of question, and each one fits a different kind of code.

| Type | Answer | Use it when |
|---|---|---|
| `noul` | the probability of yes | a passage either has a property or does not |
| `choice` | a probability for each option we define | each passage belongs in exactly one of several categories |
| `score` | a position on an ordered scale we define | the property comes in degrees |

The `instructions` give the question. For `choice` and `score`, the `criteria` define each option
or level, and those descriptions are the only definitions Laya has.

## Setup

The service address and your key go in a file called `.env` in your project folder, which should
never be committed:

```
LAYA_URL=https://laya.carlboettiger.info
LAYA_API_KEY=your-key-here
```

`load_dotenv()` reads that file into environment variables, and `decide()` sends a list of passages
with one coding scheme.

In [1]:
import os
import re
import pathlib
import requests
import pandas as pd
from dotenv import load_dotenv

load_dotenv()
laya_url = os.environ["LAYA_URL"]
laya_auth = {"Authorization": f"Bearer {os.environ['LAYA_API_KEY']}"}

def decide(texts, scheme):
    response = requests.post(f"{laya_url}/v1/systemone/batch", headers=laya_auth,
                             json={"states": texts, "questions": scheme}, timeout=300)
    response.raise_for_status()
    return [result["answers"] for result in response.json()["results"]]

## A passage

Our example is the [final rule](https://www.federalregister.gov/d/2026-14636) that listed the
Florida Keys mole skink as endangered in July 2026. We save its text to a file and split it into
paragraphs at blank lines.

In [2]:
rules = pathlib.Path("rules")
rules.mkdir(exist_ok=True)

def download(document_number):
    path = rules / f"{document_number}.txt"
    if not path.exists():
        meta = requests.get(f"https://www.federalregister.gov/api/v1/documents/{document_number}.json",
                            params={"fields[]": "raw_text_url"}, timeout=60).json()
        path.write_text(requests.get(meta["raw_text_url"], timeout=60).text)
    return path

def paragraphs(path):
    text = re.sub(r"<[^>]+>", "", path.read_text())
    paras = [re.sub(r"\s+", " ", p).strip() for p in re.split(r"\n\s*\n", text)]
    return [p for p in paras if len(p) > 200]

skink = paragraphs(download("2026-14636"))
example = next(p for p in skink if p.startswith("Sea level rise, more frequent tidal flooding"))
print(example)

Sea level rise, more frequent tidal flooding (increase of tides above the mean high tide), and increasing intensity of storm events (such as hurricanes) are the predominant threats to the Florida Keys mole skink and its habitat. Other threats to the Florida Keys mole skink include habitat loss and degradation that result from development and habitat disturbance. We also evaluated existing regulatory mechanisms and ongoing conservation measures. In the SSA report, we considered additional threats: overutilization due to recreational, educational, and scientific use; disease; oil spills; saltwater intrusion; and nonnative species. We concluded that, as indicated by the best available scientific and commercial information, these additional threats are currently having little to no impact on the Florida Keys mole skink, and thus their overall effect now and into the future is expected to be minimal. For full descriptions of all threats and how they impact the Florida Keys mole skink, pleas

## One code, three ways

Here we code the same two ideas, a climate threat and an action to address it, with different
question types. Each version records something different about the paragraph.

In [3]:
scheme = {
    "threat_noul": {"type": "noul",
                    "instructions": "Does this paragraph say that climate change threatens the species?"},
    "threat_choice": {"type": "choice",
                      "instructions": "Which threat does this paragraph describe as the most important?",
                      "criteria": {"climate": "sea level rise, flooding, drought, heat or storms",
                                   "development": "habitat loss from building, roads or land conversion",
                                   "other": "any other threat",
                                   "none": "the paragraph does not describe a threat"}},
    "threat_score": {"type": "score",
                     "instructions": "How much weight does this paragraph give to climate-related threats?",
                     "criteria": ["Not mentioned.", "Mentioned in passing.",
                                  "One of several important threats.", "The main threat."]},
    "action_noul": {"type": "noul",
                    "instructions": "Does this paragraph describe an action to reduce a climate-related threat?"},
    "action_score": {"type": "score",
                     "instructions": "How specific are the actions this paragraph describes to reduce climate-related threats?",
                     "criteria": ["No actions.", "Actions in general terms.",
                                  "Specific actions.", "Specific actions with a schedule or a responsible agency."]},
}

def summarize(answer):
    if answer["type"] == "noul":
        return round(answer["noul"], 2)
    if answer["type"] == "choice":
        return answer["choice"]
    return round(answer["score"], 2)

answers = decide([example], scheme)[0]
pd.Series({name: summarize(answer) for name, answer in answers.items()})

threat_noul             0.86
threat_choice    development
threat_score            2.28
action_noul             0.08
action_score             1.6
dtype: object

The versions do not agree. The `noul` question finds a climate threat, while the `choice`
question names development as the most important threat, and the two action questions point in
different directions. Each answer follows from how we wrote its question and options. Every answer
also carries its full probabilities, which show how close the other options came:

In [4]:
answers["threat_choice"]["probabilities"]

{'climate': 0.2644, 'development': 0.3958, 'other': 0.2407, 'none': 0.0991}

## A folder of rules

The same scheme applies to every paragraph in every file in a folder. We add a second rule, which
lists the Peñasco least chipmunk, and code each paragraph that mentions a climate-related term.

In [5]:
download("2024-28338")

rows = []
for path in sorted(rules.glob("*.txt")):
    paras = [p for p in paragraphs(path) if re.search(r"climat|sea level|flood|storm|drought", p, re.I)]
    for text, answer in zip(paras, decide(paras, scheme)):
        rows.append({"rule": path.stem, "text": text} |
                    {name: summarize(a) for name, a in answer.items()})
coded = pd.DataFrame(rows)
coded.groupby("rule")[["threat_noul", "threat_score", "action_noul", "action_score"]].mean().round(2)

,threat_noul,threat_score,action_noul,action_score
rule,,,,
2024-28338,0.17,1.99,0.21,1.75
2026-14636,0.44,2.10,0.22,1.53


## Questions

- Which of these questions comes closest to what Delach et al. coded by hand, and what would you
  change to bring it closer?
- Read the paragraphs in `coded` with the highest `threat_noul`. Does each one meet your own
  definition of a threat finding? If not, is the cause the wording, the question type, or the
  passage you sent?
- What changes if you send a sentence or a whole section in place of a paragraph?

See the [Laya documentation](https://nandhakishorm.github.io/laya/) and the
[Jev documentation](https://docs.typesafe.ai/primitives) for the full question format.